# SAKSHI: unimodal leave-one-event-out baselines

Train B1 text-only or B2 image-only with one CrisisMMD event held out. The configured modality supplies its own labels and input stream. The frozen LOEO evaluator scores the held-out event only. Keep the input dataset and notebook private.


## 1. Parameters

Choose `CONFIG_PATH` as `configs/classifier/b1_text.yaml` or `configs/classifier/b2_image.yaml`; choose one `FOLD = "loeo_<event>"` and keep `SEED = 0`. Each combination is an independent run.


In [ ]:
from pathlib import Path

BUNDLE_ROOT = None  # Set Path("/kaggle/input/<dataset>/<bundle-folder>") if discovery is ambiguous.
WORK_ROOT = Path("/kaggle/working/sakshi")
CONFIG_PATH = "configs/classifier/b1_text.yaml"
SEED = 0
FOLD = "loeo_hurricane_harvey"
RESUME_RUN = None
FINAL_EVALUATION = True
INCLUDE_RESUME_IN_EXPORT = False


## 2. Verify the bundle and stage its exact source snapshot

This cell uses only the Python standard library. Every source and cache file is checked
against the bundle's SHA256 inventory before installation. Prepared arrays stay on the
read-only input mount; only source is copied to the working directory.


In [ ]:
import hashlib
import json
import os
import runpy
import sys

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
WORK_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(WORK_ROOT / "hf_cache")

if "sakshi" in sys.modules:
    raise RuntimeError("Restart the kernel before staging a bundle or changing its input path.")
if BUNDLE_ROOT is None:
    candidates = sorted(Path("/kaggle/input").rglob("bundle.json"))
    if len(candidates) != 1:
        raise ValueError(f"Expected one input bundle, found {len(candidates)}; set BUNDLE_ROOT explicitly.")
    BUNDLE_ROOT = candidates[0].parent
BUNDLE_ROOT = Path(BUNDLE_ROOT).resolve()
manifest = json.loads((BUNDLE_ROOT / "bundle.json").read_text())
bootstrap_name = "source/src/sakshi/compute/bootstrap.py"
records = [entry for entry in manifest["files"] if entry["path"] == bootstrap_name]
bootstrap_file = BUNDLE_ROOT / bootstrap_name
if len(records) != 1 or hashlib.sha256(bootstrap_file.read_bytes()).hexdigest() != records[0]["sha256"]:
    raise ValueError("Bootstrap checksum mismatch or duplicate inventory entry.")
bootstrap = runpy.run_path(str(bootstrap_file))
workspace = bootstrap["prepare_workspace"](BUNDLE_ROOT, WORK_ROOT / "source")
os.environ["SAKSHI_PROCESSED_DIR"] = workspace["processed_dir"]
SOURCE_ROOT = Path(workspace["source_dir"])
print(json.dumps(workspace, indent=2))

## 3. Install against the hosted CUDA PyTorch

Keep the preinstalled PyTorch build rather than downloading a replacement CUDA stack.
The install runs before importing numerical libraries in this kernel. The exact installed
package list is saved with the run. If the installer requests a kernel restart, restart and
rerun from the top with the same input bundle.


In [ ]:
import importlib.metadata
import re
import subprocess

if sys.version_info < (3, 10):  # noqa: UP036 -- runs before package installation
    raise RuntimeError("Python 3.10 or later is required.")
try:
    torch_version = importlib.metadata.version("torch")
except importlib.metadata.PackageNotFoundError as exc:
    raise RuntimeError("Select a hosted GPU image with CUDA PyTorch preinstalled.") from exc
if tuple(map(int, re.match(r"(\d+)\.(\d+)", torch_version).groups())) < (2, 6):
    raise RuntimeError("The hosted PyTorch must be at least 2.6 for the pinned BERTweet weights.")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)
constraints = WORK_ROOT / "host-torch.txt"
constraints.write_text(f"torch=={torch_version}\n")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-e",
        f"{SOURCE_ROOT}[dev,ml]",
        "--upgrade-strategy",
        "only-if-needed",
        "--constraint",
        str(constraints),
    ],
    check=True,
)
# Editable installs may add a .pth file that this already-running kernel has not read.
sys.path.insert(0, str(SOURCE_ROOT / "src"))
import torch  # noqa: E402 -- install and set source path before numerical imports

if not torch.cuda.is_available():
    raise RuntimeError("CUDA PyTorch is unavailable; enable the GPU accelerator and restart.")
print(
    {
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
        "devices": torch.cuda.device_count(),
    }
)

## 4. Validate data and training configuration

The preflight checks cache integrity, split isolation and the official selector counts. Training labels and usable rows are selected from the configured modality. No official test label is used for training or checkpoint selection.


In [ ]:
import logging
from dataclasses import asdict, replace

from sakshi import atomic, paths
from sakshi.compute.preflight import inspect_inputs
from sakshi.config import load
from sakshi.models.train import TrainConfig

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
config = replace(load(SOURCE_ROOT / CONFIG_PATH, TrainConfig), seed=SEED, fold=FOLD)
if config.fold is None or not config.fold.startswith("loeo_") or config.model.modality not in ("text", "image"):
    raise ValueError("Set FOLD to a loeo_<event> manifest fold.")
if Path(workspace["processed_dir"]) != paths.PROCESSED:
    raise RuntimeError("Prepared input path was imported too early; restart the kernel.")
preflight = inspect_inputs(config, strict=True, scan_arrays=True)
preflight["bundle"] = workspace
preflight["config"] = asdict(config)
atomic.text(WORK_ROOT / "gpu_preflight.json", json.dumps(preflight, indent=2) + "\n")
print(json.dumps(preflight, indent=2))

## 5. Bounded CUDA smoke

Use real training rows and the configured microbatch. Verify only the selected modality and its active heads, then perform a finite optimizer update. Smoke weights are discarded before the seeded training run.


In [ ]:
import gc

from sakshi.compute.runner import training_smoke

smoke = training_smoke(config, WORK_ROOT / "gpu_smoke.json", require_cuda=True)
print(json.dumps(smoke, indent=2))
gc.collect()
torch.cuda.empty_cache()

## 6. Train and time the run

Select the best checkpoint on this run's development split. Save epoch recovery state, per-head label counts, optimizer updates, timing and peak GPU memory.


In [ ]:
import shutil

from sakshi.models.train import run_training

RUN_DIR = run_training(
    config,
    output_dir=WORK_ROOT / "runs/classifier",
    resume_run=Path(RESUME_RUN) if RESUME_RUN else None,
)
for name in ("gpu_preflight.json", "gpu_smoke.json"):
    shutil.copy2(WORK_ROOT / name, RUN_DIR / name)
requirements = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
atomic.text(RUN_DIR / "requirements.lock.txt", requirements)
print("Run:", RUN_DIR)
print((RUN_DIR / "training_summary.json").read_text())
print("Best dev metrics:", (RUN_DIR / "metrics_dev.json").read_text())
gc.collect()
torch.cuda.empty_cache()

## 7. Calibrate on dev

Fit temperatures and abstention thresholds using dev rows only. Review the recorded
before/after metrics and any fallback reasons. Rerunning calibration after freezing the
final test evaluation is refused by this notebook.


In [ ]:
from sakshi.models.calibrate_run import calibrate_run

if (RUN_DIR / "evaluation_loeo_request.json").exists():
    print("Evaluation already frozen; keeping the recorded calibration artifacts.")
else:
    calibration = calibrate_run(RUN_DIR)
    print(json.dumps(calibration, indent=2))
gc.collect()
torch.cuda.empty_cache()

## 8. Frozen held-out-event evaluation

Freeze model, calibration and input hashes before predicting the held-out event. B1 scores informative and humanitarian; B2 also scores gold-gated and predicted-gated severity. The event never enters training or development.


In [ ]:
from sakshi.models.evaluate_loeo import evaluate_loeo_run

if FINAL_EVALUATION:
    metrics_path = evaluate_loeo_run(RUN_DIR, bootstrap=1000)
    print(metrics_path.read_text())
else:
    print("Held-out event has not been scored. Dev-trained artifacts are ready for review.")
gc.collect()
torch.cuda.empty_cache()


## 9. Export and save

Export the best model and its metadata, calibration and any completed final metrics.
Parts are at most 768 MiB and each has a SHA256 checksum. Enable `INCLUDE_RESUME_IN_EXPORT`
for portable epoch recovery; otherwise `last.pt` remains in the private notebook output.
Raw images and the input feature arrays are excluded from the result export.

**Save the notebook version with output** before ending the session. Download the complete
export folder, including `export.json` and every `.partNNN` file. Locally restore with:

```sh
sakshi-gpu restore --export /path/to/downloaded-export --output runs/classifier/<run_id>
```

On an identical rerun, select a fresh export directory below rather than overwriting a
previous artifact set. See the guide for restoring a run that includes `last.pt` on a GPU host.


In [ ]:
from datetime import datetime, timezone

from sakshi.compute.runner import export_run

export_tag = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
EXPORT_DIR = WORK_ROOT / "exports" / f"{RUN_DIR.name}-{export_tag}"
export_manifest = export_run(RUN_DIR, EXPORT_DIR, include_resume=INCLUDE_RESUME_IN_EXPORT)
print("Download folder:", EXPORT_DIR)
print(export_manifest.read_text())

## Handoff

Restore the export locally and record per-event B1/B2 metrics. B1 has no severity output because it never sees the image. Repeat for all seven events and derive late fusion from independently calibrated B1 and B2 probabilities using dev-only weight selection.
